<a href="https://colab.research.google.com/github/nil186007/datastructure-algorithm/blob/leetcode/Cyber_Attack_Simulation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### `---------------Mandatory Information to fill------------`

### Group ID:
### Group Members Name with Student ID:
1. Student 1
2. Student 2
3. Student 3
4. Student 4


`-------------------Write your remarks (if any) that you want should get consider at the time of evaluation---------------`

Remarks: ##Add here

# ***Reference Paper: ***

https://drive.google.com/file/d/1byuGKu1Pz7bN5F0jFP5iIbd3Bt-YA9uQ/view?usp=drive_link

## **Objective:**
The agent must learn the optimal attack policy π* using Deep Reinforcement Learning to maximize cumulative rewards while navigating and attacking the simulated network.




# **Environment Details:**

## **State Space :**

The state of the environment at any given time is represented by a vector (or a dictionary in code) that characterizes each target PC. The following attributes are considered (Ref. Table 1 Pg. No. 6 of the paper):
1. **Num_Ports (Integer):** Number of open ports on the target PC. Typical range would be 0-10 ports.

2. **Is_Admin_Compromised (Binary):** 0 if the agent doesn't have admin access, 1 if the agent does.

3. **Keyboard_Security_Enabled (Binary):** 0 if keyboard security (e.g., preventing keylogging) is disabled, 1 if it's enabled.

4. **Web_Creds_Present (Binary):** 0 if no stored web credentials are found, 1 if they are present.


state_vector = `[Num_Ports, Is_Admin_Compromised, Keyboard_Security_Enabled, Web_Creds_Present]`







## **Action Space:**

Action Space: The agent can choose from the following discrete actions (Ref. Table 2 Pg. No. 7 of the paper):

*	**0: "Open Port Attack"** (attempt to exploit vulnerabilities through open ports)

* **1: "Spoofing"** (attempt to impersonate an authorized PC via IP address spoofing)

* **2: "Keylogging"** (attempt to capture credentials via keylogging, only effective if Keyboard_Security_Enabled is 0)

* **3: "Access Web Credential"** (attempt to extract credentials stored in web browsers, only effective if Web_Creds_Present is 1)


## **Rewards**
(Ref. Pg. No. 7, Section 3.2.3 of the paper):
* **+1.0:** If the action successfully acquires administrative privileges on a target PC that wasn't previously compromised.

* **-1.0:** If the action fails to acquire credentials or is invalid in the current state (e.g., attempting keylogging when Keyboard_Security_Enabled is 1).




# **Synthetic Dataset Creation:**

To make the agent learn on the cyber-attack simulated network environment, it is required to create a synthetic dataset containing of all the possible information about the network and possible actions to perform along with the reward and next_state. The agent's objective is to launch an attack on the PCs within the network, which consists of five systems labeled as PC1, PC2, PC3, PC4, and PC5. These PCs will be considered as the target PCs *(Ref. Pg. No. 8 of the paper)*.

The dataset will contain 6 columns: target PC, current_state, action, reward, next_state and done. The each row in the dataset can be seen as a `tuple : (state, action, reward, next_state, done)` for every target PC.

The state column contains a vector of 4 values with each value randomly generated: `[Num_Ports, Is_Admin_Compromised, Keyboard_Security_Enabled, Web_Creds_Present]`. The possible values for each state value are discussed in the above state space section.

Deciding the reward for each state and action value can be generated as follows:

***Probability for the success of each action:***

1. **Open Port Attack:** This action is only possible if the target PC has any open ports. If open ports exist, there's a 40% chance of success, resulting in administrative compromise and a reward of +1.0. Otherwise, it fails with a reward of -1.0. If there are no open ports to begin with, this action automatically fails, earning the agent a reward of -1.0.
2. **Spoofing:** This action cannot be performed if the target PC is already compromised. Otherwise, there is a 10% chance of success, resulting in administrative compromise and a reward of +1.0. On a failure, the reward is -1.0.
3. **Keylogging:** Keylogging will only work if Keyboard Security is disabled on the target PC. If keyboard security is disabled, the agent has an 80% chance of success, achieving administrative compromise and a reward of +1.0. This attempt automatically fails if keyboard security is enabled, earning the agent a reward of -1.0.
4. **Access Web Credentials:** Attempting to access web credentials is only possible if such credentials exist on the target PC. If credentials exist, the agent has a 60% chance of success, achieving administrative compromise and earning a reward of +1.0. This attempt automatically fails if web credentials are not present, earning the agent a reward of -1.0.

**Sample Dataset Structure (Create a dataset of  2000 rows):**


target PC | current_state | action | reward | next_state | done
--- | --- | --- | --- | --- | ---
PC2 | [2, 0, 1, 1] | keylogging | -1.0 | [2, 0, 1, 1] | False
PC2 | [2, 1, 1, 1] | keylogging | -1.0 | [2, 1, 1, 1] | False
PC2 | [2, 0, 0, 1] | keylogging | 1.0 | [2, 1, 0, 1] | False



Example scenario for creating a dataset:

Input: (target PC, state, action)

target PC | current_state | action | reward | next_state | done
--- | --- | --- | --- | --- | ---
PC2 | [2, 0, 1, 1] | keylogging |  |  |



**Target PC:** PC2

**State Vector (Randomly Generated):** [2, 0, 1, 1] (Num_Ports = 2
Is_Admin_Compromised = 0 (Not yet compromised)
Keyboard_Security_Enabled = 1 (Keyboard security is enabled)
Web_Creds_Present = 1 (Web credentials are present))

**Action (Randomly Generated):** Action 2 = "Keylogging"


Then follow the below logic for deciding the reward:

```
      if action == 2: #Keylogging
    if state[target_pc]["is_admin_compromised"] == 1:
      reward = -1.0

    elif state[target_pc]["keyboard_security_enabled"] == 0: #security enabled
      if random_number < 0.8:  #Probability condition
  new_state[target_pc]["is_admin_compromised"] = 1
        reward = 1.0
      else:
          reward = -1.0 #Returns back
    else:
          reward = -1.0
```



**Explanation:** The motive of this action by agent is to compromise the target PC. If the PC is already comparomised and the agent is still selecting an action of “Keylogging” then the reward will be -1.0. Otherwise if the PC is not compromised and “Keylogging” is disabled, then there is a possibility of 0.8 that the agent is successful in “Keylogging” and the compromised_PC flag is set to 1. Else if the agent is not successful then the reward will be -1.0.

Hence, next_state will be: [2, 0, 1, 1] -> unchanged

**Output:** (reward, next state, done)

target PC | current_state | action | reward | next_state | done
--- | --- | --- | --- | --- | ---
PC2 | [2, 0, 1, 1] | keylogging | -1.0 | [2, 0, 1, 1] | False




**Stopping condition for Agent:**

Done = True (when all the PCs are compromised then the program will get terminated)

`**OR**`

Maximum iterations reached (e.g. max_iteration = 3000)


## Requirements and Deliverables:

Implement the Traffic Flow Optimization Problem for the given above scenario for all the below mentioned RL methods.

### Create Dataset (1 Mark)

In [ ]:
# Write your code below this line to create a dataset
#-----------------------------------------------------


Display the dataset (0.5 Marks)

In [ ]:
# Write your code below this line to create a dataset
#-----------------------------------------------------

### Create a CyberAttack Environment   (0.5 Mark)

In [ ]:
# Write your code below this line to create a dataset
#-----------------------------------------------------


## Implement DQN algorithm:


### Define all the Parameters: (1 Mark)
* Number of episodes
* Max capacity of replay memory
* Batch size
* Period of Q target network updates
* Discount factor for future rewards
* Learning rate of ADAM optimizer, and etc.

In [ ]:
# Write your code below this line to create a dataset
#-----------------------------------------------------

### Implement a replay buffer for storing the experiences. (0.5 Marks)


In [ ]:
# Write your code below this line to create a dataset
#-----------------------------------------------------

### Design Main Network (0.5 Marks)
### Design Target Network (0.5 Marks)


In [ ]:
# Write your code below this line to create a dataset
#-----------------------------------------------------

### Training DQN (1 Mark) and Print all the Iterations (1 Mark)

In [ ]:
# Write your code below this line to create a dataset
#-----------------------------------------------------

## Implement Actor-Critic algorithm:


### Define all the Parameters: (1 Mark)
* Number of episodes
* Batch size
* Learning rate of ADAM optimizer, and etc.

In [ ]:
# Write your code below this line to create a dataset
#-----------------------------------------------------

### Design Actor Network (0.5 Marks)
### Design Critic Network (0.5 Marks)


In [ ]:
# Write your code below this line to create a dataset
#-----------------------------------------------------

### Training Actor-Critic Network (1 Mark) and Print all the Iterations (1 Mark)

In [ ]:
# Write your code below this line to create a dataset
#-----------------------------------------------------

## Plot the graph for Average Reward Obtained by all RL Algorithms. (1 Mark)



In [ ]:
# Write your code below this line to create a dataset
#-----------------------------------------------------

## Plot the graph for Average Success Rate Obtained by all RL Algorithms. (1 Mark)


In [ ]:
# Write your code below this line to create a dataset
#-----------------------------------------------------

## Conclude your assignment with your analysis consisting of at least 200 words by summarizing your findings of the assignment. (1 Mark)


`----write your analysis below this line-----`

